In [1]:
from jinja2 import optimizer
%load_ext autoreload
%autoreload 2


import genesis as gs
import logging
gs.init(logging_level=logging.WARNING, backend=gs.gpu)
from buffer import Buffer
from network import Network
from make_environment import Go2WalkingEnv


[I 03/12/26 15:16:19.162 81947] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
2026-03-12 15:16:21.029 Python[8392:81947] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/3y/snyjd7qd59d_gxq1x0y5gywh0000gn/T/org.python.python.savedState


In [2]:
class Rewards:
    def __init__(self) -> None:
        pass

    def __call__(self, obs, actions, info) -> float:
        return 0


In [3]:
reward_fn = Rewards()
num_envs = 1
max_steps = 100

env = Go2WalkingEnv(
    num_envs=num_envs,
    device="mps",
    show_viewer=True,
    use_terrain=False,  # Set to True for complex terrain
    episode_length_s=20.0,
    reward_fn=reward_fn
)

[Genesis] [15:16:22] [WARNING] Viewer option 'n_rendered_envs' is deprecated and will be removed in future release. Please use 'rendered_envs_idx' instead.
[Genesis] [15:16:23] [WARNING] Interactive viewer running in main thread. It will only be responsive if a simulation is running.
[Genesis] [15:16:26] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


UNSUPPORTED (log once): POSSIBLE ISSUE: unit 4 GLD_TEXTURE_INDEX_CUBE_MAP is unloadable and bound to sampler type (Float) - using zero texture because texture unloadable


In [4]:
env.set_commands(lin_vel_x=1.0, lin_vel_y=0.0, ang_vel_yaw=0.0)
policy = Network(
    num_outputs=env.num_actions,
    num_inputs=env.num_obs,
    gamma=0.99,
    lmbda=0.0,
    epsilon=0.1,
)
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=max_steps,
    device='mps'
)

In [5]:
import torch
policy.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if hasattr(m, 'weight') else None)

Network(
  (shared): Sequential(
    (0): Linear(in_features=48, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
  )
  (actor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (actor_mean): Linear(in_features=32, out_features=12, bias=True)
  (critic): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [6]:
"""num_updates = 10
num_steps_per_update = 100
device = 'mps'
policy.to(device)

for update_idx in range(num_updates):
    obs = env.reset()
    print(obs)
    for _ in range(num_steps_per_update):

        obs = obs.to(device)
        actions, value = policy.get_actions(obs)
        log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)

        obs, reward, done, info = env.step(actions)


        buffer.add_step(obs, actions, log_probs, reward, done, value)


"""


"num_updates = 10\nnum_steps_per_update = 100\ndevice = 'mps'\npolicy.to(device)\n\nfor update_idx in range(num_updates):\n    obs = env.reset()\n    print(obs)\n    for _ in range(num_steps_per_update):\n\n        obs = obs.to(device)\n        actions, value = policy.get_actions(obs)\n        log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)\n\n        obs, reward, done, info = env.step(actions)\n\n\n        buffer.add_step(obs, actions, log_probs, reward, done, value)\n\n\n"

In [ ]:
optim = torch.optim.Adam(policy.parameters(), lr=1e-3, eps=1e-8)
torch.autograd.set_detect_anomaly(True)
num_updates = 1000
steps_per_update = 2048
update_epochs = 10
minibatch_size = 128
#max_length =
device ='mps'
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=steps_per_update,
    device='mps'
)

for i in range(num_updates):
    #collect rollout
    buffer.reset()
    obs = env.reset()
    buffer.init_obs(obs, policy.get_value(obs))
    with torch.no_grad():
        for step in range(steps_per_update):
            obs = obs.to(device)
            actions, value = policy.get_actions(obs)
            log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)
            next_obs , reward, done, info = env.step(actions)
            buffer.add_step(next_obs, actions, log_probs, reward, done, value)

            obs = next_obs

            if done.any():
                obs = env.reset()

        buffer.compute_returns_and_advantages(gamma=0.99, lmbda=0.95)

    for epoch in range(update_epochs):
        batch = buffer.get_batch(minibatch_size)

        log_probs_new, values_new, entropy = policy.compute_log_probs(batch['obs'], batch['actions'])
        #print(batch)
        critic_loss, actor_loss = policy.compute_loss(states= batch['obs'],
                                                      actions= batch['actions'],
                                                      advantages=batch['advantages'],
                                                      critic_targets=batch['values'],
                                                      log_probs_old=batch['log_probs'],
                                                      returns = batch['returns'],)
        entropy_loss = -0.01 * entropy.mean()

        loss = actor_loss + 0.5 * critic_loss + entropy_loss

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5) # gradient clipping
        optim.step()

        if i % 10 == 0:
            print(f"Update {i}: Loss={loss.item():.3f}, AvgRew={buffer.rewards.mean():.3f}")






Update 0: Loss=19.468, AvgRew=0.000
Update 0: Loss=2486.238, AvgRew=0.000
Update 0: Loss=19407.773, AvgRew=0.000
Update 0: Loss=810.201, AvgRew=0.000
Update 0: Loss=30379.320, AvgRew=0.000
Update 0: Loss=1181.998, AvgRew=0.000
Update 0: Loss=642.312, AvgRew=0.000
Update 0: Loss=507.340, AvgRew=0.000
Update 0: Loss=126.149, AvgRew=0.000
Update 0: Loss=11366.456, AvgRew=0.000
